In [ ]:
import os
import gc
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam


DATASET = "fashion_mnist"  # or "mnist"
BATCH_SIZE = 128
EPOCHS = 50
noise_dim = 100
BUFFER_SIZE = 60000


if DATASET == "mnist":
    (train_images, _), (_, _) = tf.keras.datasets.mnist.load_data()
else:
    (train_images, _), (_, _) = tf.keras.datasets.fashion_mnist.load_data()

train_images = train_images.astype("float32")
train_images = (train_images - 127.5) / 127.5
train_images = np.expand_dims(train_images, axis=-1)

train_dataset = (
    tf.data.Dataset.from_tensor_slices(train_images)
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
)

print("Dataset:", DATASET)
print("Shape:", train_images.shape)


def make_generator():
    model = models.Sequential([
        layers.Input(shape=(noise_dim,)),

        layers.Dense(7 * 7 * 256, use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Reshape((7, 7, 256)),

        layers.Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(1, 5, strides=1, padding="same", activation="tanh")
    ])
    return model

generator = make_generator()
generator.summary()


def make_discriminator():
    model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),

        layers.Conv2D(64, 5, strides=2, padding="same"),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dropout(0.3),

        layers.Conv2D(128, 5, strides=2, padding="same"),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1, activation="sigmoid")
    ])
    return model

discriminator = make_discriminator()
discriminator.summary()


cross_entropy = tf.keras.losses.BinaryCrossentropy()

def d_loss(real, fake):
    real_loss = cross_entropy(tf.ones_like(real), real)
    fake_loss = cross_entropy(tf.zeros_like(fake), fake)
    return real_loss + fake_loss

def g_loss(fake):
    return cross_entropy(tf.ones_like(fake), fake)

gen_opt = Adam(2e-4, beta_1=0.5)
disc_opt = Adam(2e-4, beta_1=0.5)


@tf.function
def train_step(images):
    noise = tf.random.normal([tf.shape(images)[0], noise_dim])

    with tf.GradientTape() as g_tape, tf.GradientTape() as d_tape:
        fake_images = generator(noise, training=True)

        real_out = discriminator(images, training=True)
        fake_out = discriminator(fake_images, training=True)

        gen_loss = g_loss(fake_out)
        disc_loss = d_loss(real_out, fake_out)

    g_grad = g_tape.gradient(gen_loss, generator.trainable_variables)
    d_grad = d_tape.gradient(disc_loss, discriminator.trainable_variables)

    gen_opt.apply_gradients(zip(g_grad, generator.trainable_variables))
    disc_opt.apply_gradients(zip(d_grad, discriminator.trainable_variables))

    return gen_loss, disc_loss


def generate_images(epoch, seed):
    preds = generator(seed, training=False)

    plt.figure(figsize=(4, 4))
    for i in range(preds.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow(preds[i, :, :, 0] * 127.5 + 127.5, cmap="gray")
        plt.axis("off")

    plt.suptitle(f"Epoch {epoch}")
    plt.show()


seed = tf.random.normal([16, noise_dim])

g_history, d_history = [], []

for epoch in range(EPOCHS):
    g_losses, d_losses = [], []

    for batch in train_dataset:
        g, d = train_step(batch)
        g_losses.append(g.numpy())
        d_losses.append(d.numpy())

    avg_g = np.mean(g_losses)
    avg_d = np.mean(d_losses)

    g_history.append(avg_g)
    d_history.append(avg_d)

    print(f"Epoch {epoch+1}/{EPOCHS} | G: {avg_g:.4f} | D: {avg_d:.4f}")

    if (epoch + 1) % 10 == 0:
        generate_images(epoch + 1, seed)

    gc.collect()


generate_images(EPOCHS, seed)

plt.plot(g_history, label="Generator Loss")
plt.plot(d_history, label="Discriminator Loss")
plt.legend()
plt.title("DCGAN Loss")
plt.show()

noise = tf.random.normal([25, noise_dim])
imgs = generator(noise, training=False)

plt.figure(figsize=(6, 6))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(imgs[i, :, :, 0] * 127.5 + 127.5, cmap="gray")
    plt.axis("off")

plt.suptitle("Generated Images")
plt.tight_layout()
plt.show()

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Dataset: fashion_mnist
Shape: (60000, 28, 28, 1)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 12544)          │     1,254,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 12544)          │        50,176 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 7, 7, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose                │ (None, 14, 14, 128)    │       819,200 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_1              │ (None, 28, 28, 64)     │       204,800 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 28, 28, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_2              │ (None, 28, 28, 1)      │         1,601 │
│ (Conv2DTranspose)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,330,945 (8.89 MB)

 Trainable params: 2,305,473 (8.79 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 14, 14, 64)     │         1,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 7, 7, 128)      │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │         6,273 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 212,865 (831.50 KB)

 Trainable params: 212,865 (831.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50 | G: 0.6851 | D: 1.3720
